
# Data Hygiene & Merge Validation — **Operational Reference**

This notebook ensures your merged Reddit data is **truthful and stable** over time.
It provides:
- exact **JSON schema** expectations,
- **merge rules** (sparse histories, timestamp fallbacks, deletion thresholds),
- **integrity assertions** and **repair utilities**,
- a small **unit-test harness** to prevent regressions.


## 1. Expected JSON Schema (merged file)


**Root**
- `post: { id, title, author, score_history: [[score, ts]...], text_history?: [[text, ts]...] }`
- `comments: [ comment ]`

**Comment**
- `id, parent_id, author`
- `score_history: [[score|None, ts]...]` (sparse; never zero-fill; None only for deletion audit marker)
- `text_history: [[text, ts]...]` (append only on actual edit)
- `deleted: bool` (default False), `deleted_at?: ts` (set once at threshold)
- `replies: [ comment ]` (recursive)


## 2. Merge Rules (must not regress)


- **No scrape → no entry** for comment score history (sparse truth).
- **Timestamps**: metadata.scraped_at → filename → mtime; all **UTC** (Z-suffixed).
- **Deletion**: mark after **3 consecutive absences** (+ one-time audit marker `[None, ts]`).
- **Post root**: `len(post.score_history) == #raw_scrapes`.


## 3. Integrity Checks

In [ ]:

import os, json, re
from datetime import datetime, timezone
import pandas as pd

def parse_iso_utc(s: str):
    if not s: return None
    s = s.replace('Z', '+00:00')
    try:
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except Exception:
        # fallback compact
        m = re.search(r"(\d{8})[T_]?(\d{6})", s)
        if m:
            return datetime.strptime(m.group(1)+m.group(2), "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)
        return None

def ts_from_filename(fn: str):
    m = re.search(r"scrape_\d+_(\d{8})_(\d{6})\.json$", fn)
    if not m: return None
    return datetime.strptime(m.group(1)+m.group(2), "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)

def validate_post_dir(post_dir: str):
    merged_path = os.path.join(post_dir, "final_merged.json")
    raw_dir = os.path.join(post_dir, "raw_scrapes")
    if not os.path.exists(merged_path): 
        return None
    with open(merged_path, "r") as f:
        merged = json.load(f)
    root_ts = [parse_iso_utc(ts) for _, ts in merged.get("post",{}).get("score_history",[]) if parse_iso_utc(ts)]
    raw_ts = []
    if os.path.isdir(raw_dir):
        for fn in sorted(os.listdir(raw_dir)):
            if fn.endswith(".json"):
                t = ts_from_filename(fn)
                if t: raw_ts.append(t)
    return {
        "post_id": os.path.basename(post_dir),
        "raw_scrapes": len(raw_ts),
        "merged_root_ts": len(root_ts),
        "missing_in_merged": int(len(set(raw_ts) - set(root_ts))),
        "extra_in_merged": int(len(set(root_ts) - set(raw_ts))),
        "first_ts": min(root_ts).isoformat() if root_ts else "",
        "last_ts": max(root_ts).isoformat() if root_ts else "",
    }

def audit_tree(node, issues, path="post"):
    # Check histories are monotonic and sparse
    sh = node.get("score_history", [])
    prev = None
    for val, ts in sh:
        t = parse_iso_utc(ts)
        if prev and t and t < prev:
            issues.append((path, "non_monotonic_score_history", ts))
        prev = t or prev
    # Ensure None only appears for deletion audit
    if any(val is None for val, _ in sh):
        if not node.get("deleted", False):
            issues.append((path, "none_value_without_deleted_flag", ""))
    # Recurse
    for i, ch in enumerate(node.get("replies", []) or []):
        audit_tree(ch, issues, f"{path}.replies[{i}]")


## 4. Batch Validator & Report

In [ ]:

def validate_dataset(root_dir: str):
    rows, issues = [], []
    for pid in sorted(os.listdir(root_dir)):
        pdir = os.path.join(root_dir, pid)
        if not os.path.isdir(pdir): continue
        row = validate_post_dir(pdir)
        if row: rows.append(row)
        # audit comment tree
        merged_path = os.path.join(pdir, "final_merged.json")
        if os.path.exists(merged_path):
            with open(merged_path, "r") as f:
                data = json.load(f)
            # audit root + tree
            audit_tree(data.get("post", {}), issues, "post")
            for i, c in enumerate(data.get("comments", []) or []):
                audit_tree(c, issues, f"comments[{i}]")
    df = pd.DataFrame(rows).sort_values("post_id")
    issues_df = pd.DataFrame(issues, columns=["path","issue","detail"])
    return df, issues_df

# Example usage (set your path):
# integrity_df, issues_df = validate_dataset('/mnt/data/temporal_test')
# integrity_df.head(), issues_df.head()


## 5. Repair Utilities (optional)

In [ ]:

def repair_none_without_deleted(data: dict) -> int:
    fixed = 0
    def walk(n):
        nonlocal fixed
        sh = n.get("score_history", [])
        if any(val is None for val, _ in sh) and not n.get("deleted", False):
            # Remove trailing None markers
            n["score_history"] = [(v,t) for (v,t) in sh if v is not None]
            fixed += 1
        for ch in n.get("replies", []) or []:
            walk(ch)
    walk(data.get("post", {}))
    for c in data.get("comments", []) or []:
        walk(c)
    return fixed
